In [13]:
from pathlib import Path
import numpy as np  
import pandas as pd

#Create a data folder for the files generated by this notebook.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Data folder is ready: {DATA_DIR.resolve()}")

Data folder is ready: C:\Users\User\Documents\GitHub\AI-and-ML-Laboratory\data


In [14]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import pandas as pd

url = "https://www.rottentomatoes.com/m/star_wars_the_mandalorian_and_grogu"

# url = "https://www.rottentomatoes.com/m/mile_end_kicks"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}
resp = requests.get(url, headers=headers, timeout=10)
if resp.status_code != 200:
    print("Error HTTP:", resp.status_code)
else:
    soup = BeautifulSoup(resp.content, "html.parser")


###################### Título (fallbacks) ################################
    title = None
    h1 = soup.find("h1")
    if h1:
        title = h1.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            title = og["content"]
    print(f"Título: {title}")  


######################## Critics And summary ########################

script = soup.find("script", id="media-scorecard-json")

if script and script.string:
    data = json.loads(script.string)

    summary = data.get("description")

    tomatometer = data.get("criticsScore", {}).get("scorePercent")
    popcornmeter = data.get("audienceScore", {}).get("scorePercent")

    print("Resumen:", summary)
    print("Tomatometer:", tomatometer)
    print("Popcornmeter:", popcornmeter)


################## Platform Names ######################################
page_text = soup.get_text(" ", strip=True)

platform_names = []

for li in soup.find_all("li", attrs={"data-qa": "movies-at-home-item"}):
    a = li.find("a", href=True)

    if a and "affiliates:" in a["href"]:
        name = a.get_text(strip=True)
        platform_names.append(name)

platform_names = list(dict.fromkeys(platform_names))

print(platform_names)


####################### Critics consensus #############################

critics_consensus = None

# Opción 1: buscar por id
consensus_div = soup.find("div", id="critics-consensus")

if consensus_div:
    p = consensus_div.find("p")
    if p:
        critics_consensus = p.get_text(" ", strip=True)



print("Critics Consensus:", critics_consensus, '\n')

################# Dictionary of each movie to create my Dataset
rows = []

movie_row = {
    "title": title,
    # "url": url, 
    "tomatometer": tomatometer,
    "popcornmeter": popcornmeter,
    "summary": summary,
    "critics_consensus": critics_consensus,
    "platforms": ", ".join(platform_names)
}

#########Creating the dataframe
print(movie_row)
rows.append(movie_row)
df = pd.DataFrame(rows)
df


Título: Star Wars: The Mandalorian and Grogu
Resumen: The evil Empire has fallen, and Imperial warlords remain scattered throughout the galaxy. As the fledgling New Republic works to protect everything the Rebellion fought for, they have enlisted the help of legendary Mandalorian bounty hunter Din Djarin (Pedro Pascal) and his young apprentice Grogu.
Tomatometer: 62%
Popcornmeter: 89%
['Fandango at Home', 'Netflix', 'Apple TV', 'Prime Video']
Critics Consensus: Bountiful in action but threadbare in narrative thrust with its episodic structure, this Star Wars is more of a skirmish that coasts on the charm of its central dynamic duo. 

{'title': 'Star Wars: The Mandalorian and Grogu', 'tomatometer': '62%', 'popcornmeter': '89%', 'summary': 'The evil Empire has fallen, and Imperial warlords remain scattered throughout the galaxy. As the fledgling New Republic works to protect everything the Rebellion fought for, they have enlisted the help of legendary Mandalorian bounty hunter Din Djarin

,title,tomatometer,popcornmeter,summary,critics_consensus,platforms
0,Star Wars: The Mandalorian and Grogu,62%,89%,"The evil Empire has fallen, and Imperial warlo...",Bountiful in action but threadbare in narrativ...,"Fandango at Home, Netflix, Apple TV, Prime Video"


In [15]:
# Path for files pkl wich are smaller than JSON files
# This File will contain the dataset of Series
pickle_path = DATA_DIR / "series_urls_pages.pkl"

In [ ]:
## Check the values per page to check how many pages will be scrolled
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_movies_urls = []

for page in range(1,10):  # 1 hasta 10
    url = f"https://www.rottentomatoes.com/browse/tv_series_browse/?page={page}"
    # print(f"Scrapeando página {page}: {url}")

    resp = requests.get(url, headers=headers, timeout=10)

    if resp.status_code != 200:
        print(f"Error en página {page}: HTTP {resp.status_code}")
        continue

    soup = BeautifulSoup(resp.content, "html.parser")

    page_movies = []

    for a in soup.find_all("a", href=True):
        full_url = urljoin(url, a["href"])

        if "rottentomatoes.com/tv/" in full_url:
            page_movies.append(full_url)

    page_movies = list(dict.fromkeys(page_movies))
    print(f"Películas encontradas en página {page}: {len(page_movies)}")

    all_movies_urls.extend(page_movies)

    time.sleep(1)

all_movies_urls = list(dict.fromkeys(all_movies_urls))

##### Summarize of all movies founded
print(f"\nTotal películas únicas: {len(all_movies_urls)}")
for movie in all_movies_urls:
    print(movie)
#At the end there are Just 154 unic values, so it could be scrolled until page 5

Películas encontradas en página 1: 42
Películas encontradas en página 2: 70
Películas encontradas en página 3: 98
Películas encontradas en página 4: 126
Películas encontradas en página 5: 154
Películas encontradas en página 6: 154
Películas encontradas en página 7: 154
Películas encontradas en página 8: 154
Películas encontradas en página 9: 154

Total películas únicas: 154
https://www.rottentomatoes.com/tv/the_boroughs/s01
https://www.rottentomatoes.com/tv/maximum_pleasure_guaranteed/s01
https://www.rottentomatoes.com/tv/mating_season/s01
https://www.rottentomatoes.com/tv/kylie/s01
https://www.rottentomatoes.com/tv/youre_killing_me/s01
https://www.rottentomatoes.com/tv/skymed/s04
https://www.rottentomatoes.com/tv/the_chi/s08
https://www.rottentomatoes.com/tv/the_boys_2019/s05
https://www.rottentomatoes.com/tv/spider_noir/s01
https://www.rottentomatoes.com/tv/off_campus/s01
https://www.rottentomatoes.com/tv/widows_bay/s01
https://www.rottentomatoes.com/tv/legends_2026/s01
https://www.r